# Import libraries

In [ ]:
from bs4 import BeautifulSoup
import chardet
from collections import defaultdict, OrderedDict
import csv
from datetime import datetime
from dateutil import tz
from importlib.metadata import version
import json
import os
from os import listdir
import pandas as pd
from pathlib import Path
import platform
import re
import requests
import sys
import time
from time import sleep
from tqdm import tqdm
# from tqdm.auto import tqdm
tqdm_bar_format = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions
from dataverse_functions import dv_metrics_functions
from dataverse_functions import github_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

# For Dataverse functions and set related variables

In [ ]:
# filePath='/Users/juliangautier/Credentials/hdv_creds.yaml'
filePath='/Users/juliangautier/Credentials/demo_creds.yaml'
credentials = common_functions.import_credentials(filePath=filePath)
installationUrl = credentials['installationURL']
apiKey = credentials['apiToken']

# headers['X-Dataverse-key'] = apiKey

# Testing

## Misc

In [ ]:
def build_records(csv_path):
    # ror_url -> {metadata_field: [identifiers]}
    grouped = OrderedDict()

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ror_url = row["ror_url"].strip()
            field = row["metadata_field"].strip()
            # dataset_identifiers is a JSON-encoded list string
            identifiers = json.loads(row["dataset_identifiers"])

            if ror_url not in grouped:
                grouped[ror_url] = OrderedDict()
            # In case the same field appears twice for a URL, merge lists
            if field in grouped[ror_url]:
                grouped[ror_url][field].extend(identifiers)
            else:
                grouped[ror_url][field] = identifiers

    # Build the final list of dicts
    records = []
    for ror_url, fields in grouped.items():
        records.append({
            "ror_url": ror_url,
            "fields": {k: v for k, v in fields.items()},
        })

    with open(f"{desktopPath}/records.json", "w") as f:
        json.dump(records, f)

    return records

In [ ]:
csv_path = os.path.join(desktopPath, 'test.csv')
records = build_records(csv_path)
print(len(records))

In [ ]:
filePath = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataverse_json_metadata_from_each_known_dataverse_installation_unzipped/CORA._Research_Data_Repository_(RDR)_2026.09.02_11.05.39/Dataverse_JSON_metadata_2026.09.02_11.05.39/doi_10.34810_DATA3521_v1.0(latest_version).json'
with open(filePath, 'r') as f1:
    datasetMetadata = f1.read()
    datasetMetadata = json.loads(datasetMetadata)

In [ ]:
citationBlock = datasetMetadata['data']['metadataBlocks']['citation']
for field in citationBlock['fields']:
    if field['typeName'] == 'author':
        authors = field['value']

for author in authors:
    authorAffiliationName = author['authorAffiliation']['expandedvalue']['termName']
    authorAffiliationRor = author['authorAffiliation']['value']

    print(f'{authorAffiliationName} ({authorAffiliationRor})')

# print(json.dumps(authors, indent=4))

In [ ]:
authorsDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/csv_files_with_metadata_from_most_known_dataverse_installations/csv_files_with_metadata_from_most_known_dataverse_installations/author(citation)_2026.08.25-2026.09.02.csv',
    # parse_dates=['dataset_publication_date'],
    usecols=lambda x: x in ['dataset_pid', 'dataset_pid_url', 'authorAffiliation'],
    # dtype={'latest_dataset_version_number': str},
    encoding='utf-8-sig'
)

renamedColumns = {'authorAffiliation': 'author_affiliation'}
authorsDf = authorsDf.rename(columns=renamedColumns)

# Create new dataframe with only authors where there's a ROR in the authorAffiliation field, using regex to match the ROR pattern (https://ror.org/ followed by 9 alphanumeric characters)
authorsWithRORAffiliationsDf = authorsDf[authorsDf['author_affiliation'].str.contains(r'https://ror\.org/[0-9a-zA-Z]{9}', regex=True, na=False)]

print(f'Count of rows in authorsWithRORAffiliationsDf: {len(authorsWithRORAffiliationsDf)}')

authorsWithRORAffiliationsDf.info()

In [ ]:
objectColumns = [
    'dataverse_installation_name',
    'dataset_pid',
    'dataset_pid_url']
typeDict = {col: str for col in objectColumns}

datasetsByInstallationsDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataset_pids_from_most_known_dataverse_installations.csv',
    usecols=lambda x: x in ['dataverse_installation_name', 'dataset_pid', 'dataset_pid_url'],
    dtype=typeDict,
    encoding='utf-8-sig'
)

# Merge two dataframes
authorsWithRORAffiliationsAndInstallationUrlsDf = authorsWithRORAffiliationsDf.merge(
    datasetsByInstallationsDf, how='left',
    on=['dataset_pid', 'dataset_pid_url'])

# Use dataverse_installations_summary file to add installation URL columns
installationUrlsDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataverse_installations_summary_2026.09.02.csv',
    usecols=lambda x: x in ['Installation_name', 'Installation_URL'],
    dtype=typeDict,
    encoding='utf-8-sig'
)

renamedColumns = {
    'Installation_name': 'dataverse_installation_name',
    'Installation_URL': 'dataverse_installation_url'
    }
installationUrlsDf = installationUrlsDf.rename(columns=renamedColumns)

# Merge two dataframes
authorAffiliationsByInstallationsDf = authorsWithRORAffiliationsAndInstallationUrlsDf.merge(
    installationUrlsDf, how='left',
    on=['dataverse_installation_name'])

authorAffiliationsByInstallationsDf = authorAffiliationsByInstallationsDf[[
    'dataverse_installation_name',
    'dataverse_installation_url',
    'dataset_pid',
    'dataset_pid_url',
    'author_affiliation'
]]

print(f'Count of rows in authorAffiliationsByInstallationsDf: {len(authorAffiliationsByInstallationsDf)}')

authorAffiliationsByInstallationsDf.head()

In [ ]:
# Get list of distinct ROR URLs
distinctRORUrlsList = authorAffiliationsByInstallationsDf['author_affiliation'].to_list()
distinctRORUrlsList = list(set(distinctRORUrlsList))
print(len(distinctRORUrlsList))